# Module 2: Training Speedups + Stability You Can Trust

Your training loop from Module 1 works, but it's leaving performance on the table.

In this module, you'll:

1. **Speed up training** with mixed precision (AMP), DataLoader tuning, and `torch.compile`
2. **Make it stable** with NaN detection, gradient monitoring, and OOM prevention
3. **Profile** to find real bottlenecks (not guessed ones)

---

In [ ]:
# --- Colab / Environment Setup (run this cell first) ---
import os, subprocess

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    if not os.path.exists("/content/pytorch-production-workshop"):
        subprocess.run(["git", "clone", "https://github.com/PacktPublishing/pytorch-production-workshop.git"], cwd="/content", check=True)
    os.chdir("/content/pytorch-production-workshop/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete — GPU:", os.environ.get("COLAB_GPU", "not detected"))

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import time
import torch
import torch.nn as nn

from src.model import build_model
from src.data import prepare_wikitext2, create_dataloaders
from src.utils import set_seed, get_device, NaNDetector
from src.evaluate import evaluate

set_seed(42)
device = get_device()
print(f"Device: {device}")

In [ ]:
# Load data and model from Module 1
train_dataset, val_dataset, _, tokenizer = prepare_wikitext2(
    vocab_size=8192, seq_len=128, tokenizer_path='../tokenizer.json'
)

config = {
    'vocab_size': tokenizer.get_vocab_size(),
    'd_model': 256, 'n_heads': 4, 'd_ff': 512,
    'n_layers': 4, 'max_seq_len': 128, 'dropout': 0.1,
}
model = build_model(config).to(device)
print(f"Model: {model.count_parameters():,} parameters")

## 2.1 DataLoader Performance

The DataLoader is often the first bottleneck. If your GPU is starving for data, training is slow even with a fast GPU.

Three key knobs:
- **`num_workers`**: Parallel data loading processes (0 = main process only)
- **`pin_memory`**: Pre-allocate page-locked memory for faster CPU→GPU transfer  
- **`persistent_workers`**: Keep worker processes alive between epochs

Let's measure the difference.

In [ ]:
def benchmark_dataloader(dataset, batch_size, num_workers, pin_memory, n_batches=100):
    """Time how long it takes to iterate through n_batches."""
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
    )
    
    # Warmup
    it = iter(loader)
    for _ in range(min(5, n_batches)):
        next(it)
    
    # Benchmark
    start = time.perf_counter()
    it = iter(loader)
    for i in range(n_batches):
        batch = next(it)
        if pin_memory and device.type == 'cuda':
            batch[0].to(device, non_blocking=True)
            batch[1].to(device, non_blocking=True)
    elapsed = time.perf_counter() - start
    
    return elapsed / n_batches * 1000  # ms per batch


configs_to_test = [
    {'num_workers': 0, 'pin_memory': False},
    {'num_workers': 2, 'pin_memory': False},
    {'num_workers': 2, 'pin_memory': True},
    {'num_workers': 4, 'pin_memory': True},
]

print(f"{'Config':<40} {'ms/batch':>10}")
print('-' * 52)
for cfg in configs_to_test:
    try:
        ms = benchmark_dataloader(train_dataset, batch_size=64, **cfg)
        print(f"workers={cfg['num_workers']}, pin_memory={cfg['pin_memory']:<5}  {ms:>10.2f}")
    except Exception as e:
        print(f"workers={cfg['num_workers']}, pin_memory={cfg['pin_memory']:<5}  FAILED: {e}")

## 2.2 Automatic Mixed Precision (AMP)

AMP runs parts of the model in float16 instead of float32. This:
- **Halves memory usage** for activations
- **Speeds up matrix multiplications** on GPUs with Tensor Cores
- Requires a **GradScaler** to prevent underflow in float16 gradients

The key insight: **not everything should be float16**. PyTorch's autocast handles this automatically.

In [ ]:
def train_epoch(model, loader, optimizer, device, use_amp=False, scaler=None, max_grad_norm=1.0):
    """Train one epoch with optional AMP."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    
    for input_ids, targets in loader:
        input_ids = input_ids.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        # AMP: wrap forward pass in autocast
        with torch.autocast(device_type=device.type, enabled=use_amp, dtype=torch.float16):
            output = model(input_ids, targets=targets)
            loss = output['loss']
        
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)  # Unscale before clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
        
        optimizer.zero_grad(set_to_none=True)
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches


# --- Compare: FP32 vs AMP ---
train_loader, _ = create_dataloaders(train_dataset, val_dataset, batch_size=64)

results = {}
for mode, use_amp in [('FP32', False), ('AMP (FP16)', True)]:
    set_seed(42)
    model_bench = build_model(config).to(device)
    opt = torch.optim.AdamW(model_bench.parameters(), lr=3e-4)
    scaler = torch.amp.GradScaler('cuda') if (use_amp and device.type == 'cuda') else None
    
    # Warmup
    train_epoch(model_bench, train_loader, opt, device, use_amp=use_amp, scaler=scaler)
    
    start = time.perf_counter()
    loss = train_epoch(model_bench, train_loader, opt, device, use_amp=use_amp, scaler=scaler)
    elapsed = time.perf_counter() - start
    
    results[mode] = {'time': elapsed, 'loss': loss}
    print(f"{mode:15s} | time: {elapsed:.1f}s | loss: {loss:.4f}")

if len(results) == 2:
    speedup = results['FP32']['time'] / results['AMP (FP16)']['time']
    print(f"\nAMP speedup: {speedup:.2f}x")

## 2.3 `torch.compile` (PyTorch 2.x)

`torch.compile` fuses operations and generates optimized kernels. It's the simplest way to speed up training on PyTorch 2.x.

Gotchas:
- First iteration is slow (compilation)
- Not all operations are supported
- Dynamic shapes trigger recompilation

In [ ]:
set_seed(42)
model_compiled = build_model(config).to(device)

# torch.compile — the one-line speedup
try:
    model_compiled = torch.compile(model_compiled)
    print("Model compiled successfully")
    
    opt = torch.optim.AdamW(model_compiled.parameters(), lr=3e-4)
    
    # First epoch is slow (compilation)
    print("Compiling (first epoch)...")
    start = time.perf_counter()
    train_epoch(model_compiled, train_loader, opt, device)
    compile_time = time.perf_counter() - start
    print(f"  First epoch (includes compilation): {compile_time:.1f}s")
    
    # Subsequent epochs are fast
    start = time.perf_counter()
    loss = train_epoch(model_compiled, train_loader, opt, device)
    fast_time = time.perf_counter() - start
    print(f"  Second epoch (compiled):            {fast_time:.1f}s | loss: {loss:.4f}")

except Exception as e:
    print(f"torch.compile not available or failed: {e}")
    print("This is fine — torch.compile requires PyTorch 2.x and may not work on all platforms.")

## 2.4 Training Stability: Detecting and Fixing Failures

Production training runs often fail silently. Three common failures:

1. **NaN loss** — usually from learning rate too high, bad data, or numerical overflow
2. **Exploding gradients** — gradient norms spike before loss goes NaN
3. **OOM errors** — batch too large, activation memory grows unexpectedly

Let's demonstrate each failure mode and its fix.

In [ ]:
# --- Demo: Exploding gradients from high learning rate ---
set_seed(42)
model_unstable = build_model(config).to(device)
opt_bad = torch.optim.SGD(model_unstable.parameters(), lr=10.0)  # Way too high!

detector = NaNDetector()

print("Training with lr=10.0 (intentionally unstable)...")
print(f"{'Step':<8} {'Loss':>12} {'Max Grad':>12} {'Status':>10}")
print('-' * 46)

model_unstable.train()
for step, (x, y) in enumerate(train_loader):
    if step >= 20:
        break
    
    x, y = x.to(device), y.to(device)
    output = model_unstable(x, targets=y)
    loss = output['loss']
    
    loss.backward()
    
    grad_stats = detector.check_gradients(model_unstable)
    is_nan = detector.check_loss(loss, step)
    status = 'NaN!' if is_nan else ('WARN' if grad_stats['max_grad'] > 10 else 'OK')
    
    print(f"{step:<8} {loss.item():>12.4f} {grad_stats['max_grad']:>12.2f} {status:>10}")
    
    if is_nan:
        print("\n>>> Loss went NaN — this is what happens without gradient clipping!")
        break
    
    opt_bad.step()
    opt_bad.zero_grad()

In [ ]:
# --- Fix: Same setup but with gradient clipping ---
set_seed(42)
model_stable = build_model(config).to(device)
opt_clipped = torch.optim.SGD(model_stable.parameters(), lr=1.0)  # Still high, but clipped

print("Training with lr=1.0 + gradient clipping (max_norm=1.0)...")
print(f"{'Step':<8} {'Loss':>12} {'Grad Norm':>12} {'Clipped?':>10}")
print('-' * 46)

model_stable.train()
for step, (x, y) in enumerate(train_loader):
    if step >= 20:
        break
    
    x, y = x.to(device), y.to(device)
    output = model_stable(x, targets=y)
    loss = output['loss']
    loss.backward()
    
    # Clip and capture the original norm
    grad_norm = torch.nn.utils.clip_grad_norm_(model_stable.parameters(), max_norm=1.0)
    clipped = 'YES' if grad_norm > 1.0 else 'no'
    
    print(f"{step:<8} {loss.item():>12.4f} {grad_norm.item():>12.2f} {clipped:>10}")
    
    opt_clipped.step()
    opt_clipped.zero_grad()

print("\n>>> Training stayed stable thanks to gradient clipping!")

In [ ]:
# --- Demo: OOM Prevention ---
# In production, you want to fail gracefully, not crash the job.

def estimate_memory(model, batch_size, seq_len, dtype=torch.float32):
    """Rough estimate of peak training memory."""
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    grad_bytes = param_bytes  # Gradients are same size as parameters
    
    # Optimizer states (Adam has 2 states per param)
    optimizer_bytes = param_bytes * 2
    
    # Activation memory (rough: ~4x model size per batch element)
    activation_bytes = batch_size * seq_len * model.d_model * 4 * model.transformer.num_layers
    
    total_mb = (param_bytes + grad_bytes + optimizer_bytes + activation_bytes) / 1024 / 1024
    return {
        'params_mb': param_bytes / 1024 / 1024,
        'grads_mb': grad_bytes / 1024 / 1024,
        'optimizer_mb': optimizer_bytes / 1024 / 1024,
        'activations_mb': activation_bytes / 1024 / 1024,
        'total_mb': total_mb,
    }

for bs in [32, 64, 128, 256, 512]:
    mem = estimate_memory(model, bs, 128)
    print(f"batch_size={bs:>4d}  →  ~{mem['total_mb']:>8.1f} MB  "
          f"(params: {mem['params_mb']:.1f}, activations: {mem['activations_mb']:.1f})")

## 2.5 PyTorch Profiler

Don't guess where bottlenecks are — **measure**. The PyTorch Profiler captures CPU and GPU time per operation.

In [ ]:
from torch.profiler import profile, ProfilerActivity, schedule, tensorboard_trace_handler

train_loader_profile, _ = create_dataloaders(train_dataset, val_dataset, batch_size=64)

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

activities = [ProfilerActivity.CPU]
if device.type == 'cuda':
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    schedule=schedule(wait=2, warmup=2, active=6, repeat=1),
    on_trace_ready=tensorboard_trace_handler('../logs/profiler'),
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    for step, (x, y) in enumerate(train_loader_profile):
        if step >= 12:  # wait(2) + warmup(2) + active(6) + buffer
            break
        x, y = x.to(device), y.to(device)
        output = model(x, targets=y)
        output['loss'].backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        prof.step()

# Print top operations by CPU time
print(prof.key_averages().table(sort_by='cpu_time_total', row_limit=15))

In [ ]:
# Export Chrome trace for detailed visual analysis
# Open chrome://tracing in Chrome and load this file
prof.export_chrome_trace('../logs/profiler/trace.json')
print("Chrome trace saved to ../logs/profiler/trace.json")
print("Open chrome://tracing and load the file for detailed analysis.")

## Key Takeaways

| Technique | When to use | Typical speedup |
|-----------|------------|----------------|
| `num_workers > 0` | GPU training | 1.5-3x |
| `pin_memory=True` | CUDA | 1.1-1.3x |
| AMP (float16) | CUDA with Tensor Cores | 1.5-2x |
| `torch.compile` | PyTorch 2.x, stable shapes | 1.2-1.5x |
| Gradient clipping | Always | N/A (stability) |
| NaN detection | Always | N/A (reliability) |

**Production rule**: Enable AMP + gradient clipping + NaN detection by default. Tune `num_workers` and `batch_size` per hardware.

**Next up**: Module 3 — optimizing inference and exporting the model for deployment.